In [30]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config import PROCESSED_DATA_DIR

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Final Dataset

In [31]:
# TODO: Load dashboard_data.parquet
# %%
# Path to final dataset
dashboard_data_path = PROCESSED_DATA_DIR / "dashboard_data.parquet"

# Load dataset
df = pd.read_parquet(dashboard_data_path)
print("Dashboard dataset shape:", df.shape)
df.head()


Dashboard dataset shape: (200000, 37)


,crash_date,crash_time,on_street_name,off_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,collision_id,vehicle_type_code1,vehicle_type_code2,borough,zip_code,latitude,longitude,location,crash_year,crash_month,crash_day,crash_day_of_week,is_weekend,crash_hour,lat_bin,lon_bin,total_injured,total_killed,severity_score,num_vehicle_types,has_truck,has_bus,has_bicycle
0,2019-11-01,09:25:00,PARSONS BOULEVARD,85 AVENUE,0,0,0,0,0,0,0,0,Unspecified,Unspecified,4233299,SEDAN,STATION WAGON/SPORT UTILITY VEHICLE,NONE,None,40.712223,-73.80650,"{'human_address': None, 'latitude': '40.712223...",2019,11,1,4,0,NaN,3,8,0,0,0,2,0,0,0
1,2016-10-06,11:40:00,AVENUE O,EAST 10 STREET,0,0,0,0,0,0,0,0,Following Too Closely,Unspecified,3535160,STATION WAGON/SPORT UTILITY VEHICLE,STATION WAGON/SPORT UTILITY VEHICLE,BROOKLYN,11230,40.612430,-73.96386,"{'human_address': None, 'latitude': '40.61243'...",2016,10,6,3,0,NaN,1,7,0,0,0,2,0,0,0
2,2016-07-11,08:30:00,EAST 125 STREET,1 AVENUE,0,0,0,0,0,0,0,0,Unspecified,Unspecified,3479651,FLAT BED,4 DR SEDAN,NONE,None,40.801754,-73.93121,"{'human_address': None, 'latitude': '40.801754...",2016,7,11,0,0,NaN,4,7,0,0,0,2,0,0,0
3,2017-10-02,00:00:00,BORDEN AVENUE,HAMILTON PLACE,1,0,0,0,0,0,1,0,Passenger Distraction,Unspecified,3763990,SEDAN,SEDAN,QUEENS,11378,40.726160,-73.90012,"{'human_address': None, 'latitude': '40.72616'...",2017,10,2,0,0,NaN,3,8,2,0,2,2,0,0,0
4,2020-03-16,13:30:00,WEST 37 STREET,None,0,0,0,0,0,0,0,0,Unspecified,Unspecified,4301217,SEDAN,PICK-UP TRUCK,NONE,None,40.755290,-73.99493,"{'human_address': None, 'latitude': '40.75529'...",2020,3,16,0,0,NaN,4,7,0,0,0,2,1,0,0


## 2. Collisions over time
Hypothesis: We expect collision counts to show strong monthly seasonality, with higher collisions in warmer months (May–October) due to increased travel and pedestrian activity. Daily patterns should show weekday peaks and lower weekend collision counts.

In [32]:
# RQ1: How do collision counts change over time (daily & monthly trends)?

# Ensure crash_date is datetime
df['crash_date'] = pd.to_datetime(df['crash_date'], errors='coerce')

# --- Daily counts ---
daily_counts = (
    df
    .groupby('crash_date')
    .size()
    .reset_index(name='collision_count')
    .sort_values('crash_date')
)

fig_daily = px.line(
    daily_counts,
    x='crash_date',
    y='collision_count',
    title="Daily Motor Vehicle Collisions in NYC",
    labels={'crash_date': 'Crash Date', 'collision_count': 'Number of Collisions'}
)
fig_daily.show()

# --- Monthly counts (year-month) ---
df['year_month'] = df['crash_date'].dt.to_period('M').astype(str)

monthly_counts = (
    df
    .groupby('year_month')
    .size()
    .reset_index(name='collision_count')
    .sort_values('year_month')
)

fig_monthly = px.line(
    monthly_counts,
    x='year_month',
    y='collision_count',
    title="Monthly Motor Vehicle Collisions in NYC",
    labels={'year_month': 'Year-Month', 'collision_count': 'Number of Collisions'}
)
fig_monthly.update_xaxes(tickangle=-45)
fig_monthly.show()


Interpretation: Analyzing daily and monthly trends reveals how crashes fluctuate over time. If the hypothesis is correct, seasonal increases may reflect behavior changes (more driving, tourism, outdoor activity), while weekday peaks suggest commuting-related risk. Any deviations from expected patterns may highlight unusual events (e.g., weather anomalies, holidays, pandemic effects) that significantly affected city-wide mobility.

## 3. Collision per year
Hypothesis: We expect collision counts to decrease in recent years, largely due to NYC safety initiatives, road redesigns, and reduced traffic during pandemic years.

In [33]:
# RQ2: Are motor vehicle collisions increasing or decreasing across recent years?

yearly_counts = (
    df
    .groupby('crash_year')
    .size()
    .reset_index(name='collision_count')
    .sort_values('crash_year')
)

fig_yearly = px.line(
    yearly_counts,
    x='crash_year',
    y='collision_count',
    markers=True,
    title="Yearly Motor Vehicle Collisions in NYC",
    labels={'crash_year': 'Year', 'collision_count': 'Number of Collisions'}
)
fig_yearly.show()


Interpretation: The year-over-year trend reveals whether overall collisions are rising or falling.
A downward trend would support the hypothesis and suggest improvements in traffic safety, while an upward trend would indicate growing traffic volume, riskier driver behavior, or ineffective safety measures.

## 4. Collision by borough
Hypothesis: We expect Manhattan and Brooklyn to show the highest collision counts because they have the highest vehicle density and congested road networks. High population and commercial activity increase traffic flow, which typically increases crash probabilities.

In [34]:
# RQ3: Which borough has the highest number of motor vehicle collisions?

borough_counts = (
    df
    .groupby('borough')
    .size()
    .reset_index(name='collision_count')
    .sort_values('collision_count', ascending=False)
)

fig_borough = px.bar(
    borough_counts,
    x='borough',
    y='collision_count',
    title="Number of Collisions by Borough",
    labels={'borough': 'Borough', 'collision_count': 'Number of Collisions'}
)
fig_borough.show()


Interpretation: Collisions were highest in Brooklyn, confirming the hypothesis. This aligns with the borough’s dense traffic patterns and heavy daily commuting. The relatively lower collisions in Staten Island match expectations due to lower population and fewer major roads. This suggests borough-level infrastructure demand correlates strongly with crash frequency.


## 5. Injuries and fatalities by borough
Hypothesis: We expect Queens and Brooklyn to show the highest injury rates due to dense traffic and pedestrian activity, while Staten Island should have the lowest because of lower population density.

In [35]:
# RQ4: How do injury and fatality rates vary across NYC boroughs?

# Aggregate injuries and fatalities by borough
severity_by_borough = (
    df
    .groupby('borough', dropna=False)[['total_injured', 'total_killed']]
    .sum()
    .reset_index()
    .fillna({'borough': 'UNKNOWN'})
)

fig_severity_borough = px.bar(
    severity_by_borough,
    x='borough',
    y=['total_injured', 'total_killed'],
    barmode='group',
    title="Injuries and Fatalities by Borough",
    labels={'value': 'Count', 'borough': 'Borough', 'variable': 'Severity Type'}
)
fig_severity_borough.show()


Interpretation: The comparison across boroughs highlights spatial disparities in road safety. If some boroughs show disproportionately high injuries or fatalities, it may indicate issues such as overcrowded roads, poor street design, or inadequate enforcement in those areas.

## 6. Collisions per hour of day
Hypothesis: Collision rates should peak during rush hours (7–9 AM and 4–7 PM) due to heavy traffic, and be lowest during late-night hours (2–5 AM).

In [36]:
import pandas as pd
import plotly.express as px

# RQ5: Which hours of the day have the highest and lowest collision rates?

hourly_counts = (
    df
    .dropna(subset=['crash_hour'])                                  # drop missing hours
    .assign(
        crash_hour=lambda d: pd.to_numeric(d['crash_hour'], errors='coerce')
    )                                                                # convert to numeric
    .dropna(subset=['crash_hour'])                                  # drop rows that couldn't convert
    .assign(
        crash_hour=lambda d: d['crash_hour'].astype(int)
    )                                                                # ensure int 0–23
    .groupby('crash_hour')
    .size()
    .reset_index(name='collision_count')
    .sort_values('crash_hour')
)

print(hourly_counts.head())  # <- check that you actually have rows

fig_hourly = px.bar(
    hourly_counts,
    x='crash_hour',
    y='collision_count',
    title="Collisions by Hour of Day",
    labels={
        'crash_hour': 'Hour of Day (0–23)',
        'collision_count': 'Number of Collisions'
    }
)

fig_hourly.show()

Empty DataFrame
Columns: [crash_hour, collision_count]
Index: []


Interpretation: Hourly patterns reveal how traffic volume and behavior affect crash risk.
A clear rush-hour spike supports the hypothesis and shows the impact of commuting patterns.
Unexpected peaks (e.g., at midnight) would suggest behavioral factors such as impaired or fatigued driving.

## 7. Top 10 contributing factors
Hypothesis: We expect “Driver Inattention” and “Following Too Closely” to appear prominently because they are common behavioral issues in dense urban driving.

In [37]:
# RQ6: What are the most common contributing factors behind NYC collisions?

factor_series = (
    df['contributing_factor_vehicle_1']
    .fillna("Unspecified")
    .value_counts()
    .nlargest(10)
)

factor_df = factor_series.reset_index()
factor_df.columns = ['factor', 'count']

fig_factors = px.bar(
    factor_df,
    x='factor',
    y='count',
    title="Top 10 Contributing Factors (Vehicle 1)",
    labels={'factor': 'Contributing Factor', 'count': 'Number of Collisions'}
)
fig_factors.update_xaxes(tickangle=-45)
fig_factors.show()


Interpretation: The top factors were Driver Inattention and failure to yield right-of-way, consistent with typical NYC driver behavior and past DOT reports. “Driver Inattention” dominated, reinforcing the need for awareness campaigns. Less common factors like “Mechanical Failure” appeared rarely, suggesting that human behavior drives most collision risk.

## 8. Pedestrian vs cyclist vs motorist injuries by borough
Hypothesis: Pedestrian and cyclist injuries should be highest in Manhattan and Brooklyn, where walking and cycling rates are highest. Motorist injuries may be more evenly distributed across boroughs.

In [38]:
# RQ7: How do pedestrian, cyclist, and motorist injuries differ across boroughs?

injury_cols = [
    'number_of_pedestrians_injured',
    'number_of_cyclist_injured',
    'number_of_motorist_injured'
]

injuries_by_borough = (
    df
    .groupby('borough', dropna=False)[injury_cols]
    .sum()
    .reset_index()
    .fillna({'borough': 'UNKNOWN'})
)

# Reshape to long format for grouped bar
injuries_melted = injuries_by_borough.melt(
    id_vars='borough',
    value_vars=injury_cols,
    var_name='injury_type',
    value_name='count'
)

injuries_melted['injury_type'] = injuries_melted['injury_type'].replace({
    'number_of_pedestrians_injured': 'Pedestrians',
    'number_of_cyclist_injured': 'Cyclists',
    'number_of_motorist_injured': 'Motorists'
})

fig_injuries = px.bar(
    injuries_melted,
    x='borough',
    y='count',
    color='injury_type',
    barmode='group',
    title="Injuries by Borough and Road User Type",
    labels={'borough': 'Borough', 'count': 'Number of Injuries', 'injury_type': 'Road User'}
)
fig_injuries.show()


Interpretation: Comparing injury types across boroughs helps identify which areas pose the greatest risks for different road users.
If the hypothesis holds, it may suggest a need for better bike lanes, pedestrian infrastructure, and traffic calming measures in highly active district

## 9. Vehicle types most frequently involved 
Hypothesis: Passenger vehicles and taxis should dominate due to their high presence on NYC roads.

In [39]:
# RQ8: Which vehicle types are most frequently involved in collisions?

vehicle_series = (
    df['vehicle_type_code1']
    .fillna("UNKNOWN")
    .value_counts()
    .nlargest(10)
)

vehicle_df = vehicle_series.reset_index()
vehicle_df.columns = ['vehicle_type', 'count']

fig_vehicles = px.bar(
    vehicle_df,
    x='vehicle_type',
    y='count',
    title="Top 10 Vehicle Types Involved in Collisions (Vehicle 1)",
    labels={'vehicle_type': 'Vehicle Type', 'count': 'Number of Involved Vehicles'}
)
fig_vehicles.update_xaxes(tickangle=-45)
fig_vehicles.show()


Interpretation: Passenger cars and Sedan appear most frequently, supporting the hypothesis. The high number of sedan aligns with urban mobility patterns, especially in Manhattan. Heavy vehicles occurred less often but may pose higher severity risk. This distribution reflects real-world vehicle presence.

## 10. Spatial density  (heatmap)
Hypothesis: Collision density will be highest in high-traffic hubs (e.g., Midtown Manhattan, Downtown Brooklyn, major bridges/tunnels) and lowest in suburban or less populated neighborhoods.

In [40]:
# RQ9: How does collision density vary geographically across NYC neighborhoods?

# Drop rows with missing coordinates
df_map = df.dropna(subset=['latitude', 'longitude'])

fig_heatmap = px.density_mapbox(
    df_map,
    lat='latitude',
    lon='longitude',
    z=None,
    radius=10,
    center=dict(lat=df_map['latitude'].mean(), lon=df_map['longitude'].mean()),
    zoom=9,
    mapbox_style='open-street-map',
    title="Spatial Density of Collisions in NYC"
)
fig_heatmap.show()


/var/folders/pg/6w5xds6d0zxdjdshsxthjxbm0000gn/T/ipykernel_2646/921715520.py:6: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



Interpretation: A spatial density map highlights city hotspots where crashes concentrate.
High-density clusters may indicate structural issues such as complex intersections, heavy tourism zones, or high-volume commuter routes.
Low-density regions reflect suburban street layouts and lighter traffic.

## 11. Contributing factors vs severity (average severity per factor)
Hypothesis: Behavioral factors (e.g., Driver Inattention, Failure to Yield, Unsafe Speed) will correlate more strongly with severe crashes than environmental factors (e.g., rain, road surface conditions).

In [41]:
# RQ10: Do environmental or behavioral contributing factors correlate with higher crash severity?

# Group by factor and compute mean severity_score
severity_by_factor = (
    df
    .dropna(subset=['contributing_factor_vehicle_1'])
    .groupby('contributing_factor_vehicle_1')['severity_score']
    .mean()
    .reset_index()
)

# Keep top 10 by mean severity_score (or top by count if you prefer)
severity_by_factor = severity_by_factor.sort_values('severity_score', ascending=False).head(10)

fig_severity_factor = px.bar(
    severity_by_factor,
    x='contributing_factor_vehicle_1',
    y='severity_score',
    title="Average Crash Severity by Contributing Factor (Top 10)",
    labels={
        'contributing_factor_vehicle_1': 'Contributing Factor',
        'severity_score': 'Average Severity Score'
    }
)
fig_severity_factor.update_xaxes(tickangle=-45)
fig_severity_factor.show()


Interpretation: If behavioral factors show a stronger association with severe crashes, it supports the hypothesis that driver actions — not weather — are the dominant contributor to high-severity collisions.
This insight can guide policy toward better enforcement, awareness campaigns, and driver education programs.

## 12. Key Insights and Dashboard Recommendations

In [42]:
# Save filtered data for dashboard
df.to_parquet(PROCESSED_DATA_DIR / "dashboard_filtered.parquet", index=False)

# Borough options
borough_options = df['borough'].dropna().unique().tolist()

# Year options
year_options = sorted(df['crash_year'].dropna().unique().tolist())

# Vehicle options: combine type_code1 and type_code2
vehicle_cols = [col for col in ['vehicle_type_code1', 'vehicle_type_code2'] if col in df.columns]

if vehicle_cols:
    vehicle_series = pd.concat([df[col].dropna() for col in vehicle_cols])
    vehicle_options = sorted(vehicle_series.unique().tolist())
else:
    vehicle_options = []

print("Borough options:", borough_options)
print("Year options:", year_options[:10], "...")
print("Vehicle options:", vehicle_options[:10], "...")


Borough options: ['NONE', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND', 'MANHATTAN']
Year options: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021] ...
Vehicle options: ['000', '11 PA', '11111', '12 FE', '2 DR SEDAN', '3 WHE', '3-DOOR', '4', '4 DR SEDAN', '4DR'] ...


**TODO: Document:**
- Key findings from analysis
- Most impactful visualizations for dashboard
- Recommended filters and interactions
- Story to tell through the dashboard

In [43]:
df.columns

Index(['crash_date', 'crash_time', 'on_street_name', 'off_street_name',
       'number_of_persons_injured', 'number_of_persons_killed',
       'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
       'number_of_cyclist_injured', 'number_of_cyclist_killed',
       'number_of_motorist_injured', 'number_of_motorist_killed',
       'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2',
       'collision_id', 'vehicle_type_code1', 'vehicle_type_code2', 'borough',
       'zip_code', 'latitude', 'longitude', 'location', 'crash_year',
       'crash_month', 'crash_day', 'crash_day_of_week', 'is_weekend',
       'crash_hour', 'lat_bin', 'lon_bin', 'total_injured', 'total_killed',
       'severity_score', 'num_vehicle_types', 'has_truck', 'has_bus',
       'has_bicycle', 'year_month'],
      dtype='object')

In [44]:
print("Rows in df:", len(df))
print(df['crash_hour'].head())
print(df['crash_hour'].value_counts().head())

Rows in df: 200000
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: crash_hour, dtype: float64
Series([], Name: count, dtype: int64)


In [45]:
print(df['crash_time'].head(20))
print(df['crash_time'].isna().mean())

0     09:25:00
1     11:40:00
2     08:30:00
3     00:00:00
4     13:30:00
5     02:30:00
6     20:55:00
7     22:48:00
8     08:45:00
9     04:00:00
10    19:30:00
11    23:52:00
12    13:21:00
13    10:00:00
14    12:50:00
15    16:20:00
16    19:00:00
17    20:30:00
18    17:00:00
19    02:58:00
Name: crash_time, dtype: object
0.0
